# Dropout Prevention - Evaluate
This notebook reloads the processed dataset and saved model, recreates the same train-test split, evaluates predictive performance, and explains the model with SHAP.


## Import Libraries

In [1]:
from pathlib import Path
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import xgboost as xgb

/Users/admin/Documents/GitHub/Student Dropout Prediction and Alert System/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the Processed Dataset and Saved Model

In [2]:
base_dir = Path.cwd().resolve()
if not (base_dir / "data").exists():
    base_dir = base_dir.parent.parent

df = pd.read_csv(base_dir / "data" / "processed" / "dropout" / "dropout_preprocessed.csv")
model = joblib.load(base_dir / "models" / "dropout" / "oula_ews_model.pkl")
features = joblib.load(base_dir / "models" / "dropout" / "model_features.pkl")

X = df[features]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:3: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:3: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
/var/folders/b3/0fd8bgg112d1_6p01j251slr0000gn/T/ipykernel_48853/3476569981.py:1: 

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\processed\\dropout_preprocessed.csv'

## Evaluate the Model on the Test Set

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Confusion Matrix:')
print(pd.crosstab(y_test, y_pred, rownames=['Actual'], colnames=['Predicted']))
print('Classification Report:')
print(classification_report(y_test, y_pred))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob):.2f}')


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Success (0)', 'At-Risk (1)'])
fig, ax = plt.subplots(figsize=(6, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('XGBoost Confusion Matrix on Test Data')
plt.show()

## Explain the Model with SHAP

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, plot_type='bar')
shap.summary_plot(shap_values, X_test)

## Estimate Stability with Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f'ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')